# Visual-Inertial Odometry with Invariant EKF

This notebook demonstrates implementing a simplified Visual-Inertial Odometry (VIO) system using the Invariant Extended Kalman Filter.

## Learning Objectives

- Understand VIO problem formulation
- Implement IEKF for SE(3) pose estimation
- Fuse IMU and visual measurements
- Compare with standard EKF approach

## Prerequisites

- Completed notebooks 01 and 02
- Understanding of SE(3) and IMU dynamics
- Knowledge of camera projection models

In [ ]:
# Install required packages
# !pip install numpy scipy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import expm, logm

np.random.seed(123)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. SE(3) Utilities

SE(3) is the Special Euclidean group representing 3D poses (position + orientation).

A pose is represented as a 4×4 matrix:
```
T = [R  t]
    [0  1]
```
where R ∈ SO(3) is rotation and t ∈ R³ is translation.

In [ ]:
def skew(v):
    """Convert 3D vector to skew-symmetric matrix."""
    return np.array([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0]
    ])

def exp_so3(omega):
    """Exponential map for SO(3)."""
    angle = np.linalg.norm(omega)
    if angle < 1e-10:
        return np.eye(3)
    axis = omega / angle
    K = skew(axis)
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * K @ K

def exp_se3(xi):
    """Exponential map for SE(3). xi = [rho, theta] where rho is translation, theta is rotation."""
    rho = xi[:3]  # translation part
    theta = xi[3:]  # rotation part
    
    R = exp_so3(theta)
    
    angle = np.linalg.norm(theta)
    if angle < 1e-10:
        V = np.eye(3)
    else:
        axis = theta / angle
        K = skew(axis)
        V = np.eye(3) + ((1 - np.cos(angle)) / angle) * K + ((angle - np.sin(angle)) / angle) * K @ K
    
    t = V @ rho
    
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T

def pose_to_matrix(R, t):
    """Convert R and t to 4x4 transformation matrix."""
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T

# Test
xi_test = np.array([0.1, 0.2, 0.3, 0.01, 0.02, 0.03])
T_test = exp_se3(xi_test)
print(f"SE(3) transformation:\n{T_test}")

## 2. VIO Problem Setup

### State
- Pose T ∈ SE(3)
- Velocity v ∈ R³
- IMU biases (simplified: assuming known)

### Measurements
- **IMU**: Angular velocity ω and linear acceleration a
- **Camera**: 2D projections of 3D landmarks

### Dynamics
```
Ṫ = T * [skew(ω)  v]
          [0       0]
v̇ = R * a + g
```

In [ ]:
# Simulation parameters
dt = 0.01  # 100 Hz
T_sim = 10.0
N = int(T_sim / dt)

# Gravity
g = np.array([0, 0, -9.81])

# Camera parameters
focal_length = 500.0
image_center = np.array([320.0, 240.0])

# Landmarks (fixed in world frame)
num_landmarks = 10
landmarks = np.random.randn(num_landmarks, 3) * 5
landmarks[:, 2] += 5  # Move forward

print(f"Simulating {N} time steps")
print(f"{num_landmarks} landmarks")

## 3. Generate Ground Truth Trajectory

In [ ]:
# Simple trajectory: constant velocity with slight rotation
omega_true = np.array([0.0, 0.1, 0.0])  # rad/s
v_body_true = np.array([1.0, 0.0, 0.0])  # m/s in body frame

# Storage
T_true = np.zeros((N, 4, 4))
v_true = np.zeros((N, 3))
T_true[0] = np.eye(4)
v_true[0] = np.array([1.0, 0.0, 0.0])

# IMU measurements (with noise)
imu_omega = np.zeros((N, 3))
imu_accel = np.zeros((N, 3))

Q_omega = 0.01 * np.eye(3)
Q_accel = 0.1 * np.eye(3)

for i in range(N):
    R_true = T_true[i, :3, :3]
    
    # IMU measurements (in body frame)
    imu_omega[i] = omega_true + np.random.multivariate_normal(np.zeros(3), Q_omega)
    
    # Acceleration in body frame (includes gravity)
    accel_body = R_true.T @ (-g)
    imu_accel[i] = accel_body + np.random.multivariate_normal(np.zeros(3), Q_accel)
    
    if i < N - 1:
        # Propagate pose
        xi = np.concatenate([v_body_true * dt, omega_true * dt])
        T_true[i+1] = T_true[i] @ exp_se3(xi)
        
        # Velocity stays constant in world frame for this simple case
        v_true[i+1] = T_true[i+1, :3, :3] @ v_body_true

print(f"Generated {N} ground truth poses")

## 4. Generate Visual Measurements

In [ ]:
def project_point(T_cam_world, point_world):
    """Project 3D point to 2D image using camera pose."""
    # Transform to camera frame
    T_world_cam = np.linalg.inv(T_cam_world)
    point_cam_h = T_world_cam @ np.append(point_world, 1)
    point_cam = point_cam_h[:3]
    
    # Check if in front of camera
    if point_cam[2] <= 0:
        return None
    
    # Project
    u = focal_length * point_cam[0] / point_cam[2] + image_center[0]
    v = focal_length * point_cam[1] / point_cam[2] + image_center[1]
    
    # Check if in image bounds
    if u < 0 or u > 640 or v < 0 or v > 480:
        return None
    
    return np.array([u, v])

# Generate visual measurements (sparse - not every frame)
visual_measurements = {}
R_visual = 1.0 * np.eye(2)  # pixel noise

for i in range(0, N, 10):  # Every 10th frame
    measurements_i = []
    for j, landmark in enumerate(landmarks):
        proj = project_point(T_true[i], landmark)
        if proj is not None:
            # Add noise
            proj_noisy = proj + np.random.multivariate_normal(np.zeros(2), R_visual)
            measurements_i.append((j, proj_noisy))
    
    if measurements_i:
        visual_measurements[i] = measurements_i

print(f"Generated visual measurements for {len(visual_measurements)} frames")
print(f"Average {np.mean([len(v) for v in visual_measurements.values()]):.1f} landmarks per frame")

## 5. Invariant EKF for VIO

The IEKF for VIO maintains:
- Pose estimate T̂ ∈ SE(3)
- Velocity estimate v̂ ∈ R³
- Covariance in tangent space

In [ ]:
class VIO_IEKF:
    def __init__(self, T0, v0, P0, Q_imu, R_vis):
        self.T = T0.copy()  # SE(3) pose
        self.v = v0.copy()  # velocity
        self.P = P0.copy()  # 9x9 covariance (6 for pose + 3 for velocity)
        self.Q = Q_imu      # IMU noise
        self.R = R_vis      # Visual measurement noise
        self.g = np.array([0, 0, -9.81])
    
    def predict(self, omega, accel, dt):
        """Predict with IMU measurements."""
        R = self.T[:3, :3]
        
        # Propagate velocity
        self.v = self.v + (R @ accel + self.g) * dt
        
        # Propagate pose
        xi = np.concatenate([self.v * dt, omega * dt])
        self.T = self.T @ exp_se3(xi)
        
        # Propagate covariance (simplified)
        self.P = self.P + self.Q * dt
    
    def update_visual(self, landmark_world, measurement_2d):
        """Update with visual measurement."""
        # Transform landmark to camera frame
        T_inv = np.linalg.inv(self.T)
        landmark_cam_h = T_inv @ np.append(landmark_world, 1)
        landmark_cam = landmark_cam_h[:3]
        
        if landmark_cam[2] <= 0:
            return  # Behind camera
        
        # Predicted measurement
        u_pred = focal_length * landmark_cam[0] / landmark_cam[2] + image_center[0]
        v_pred = focal_length * landmark_cam[1] / landmark_cam[2] + image_center[1]
        z_pred = np.array([u_pred, v_pred])
        
        # Innovation
        innovation = measurement_2d - z_pred
        
        # Measurement Jacobian (simplified - only pose)
        X, Y, Z = landmark_cam
        H_pose = np.zeros((2, 6))
        H_pose[0, 0] = focal_length / Z
        H_pose[0, 2] = -focal_length * X / (Z * Z)
        H_pose[1, 1] = focal_length / Z
        H_pose[1, 2] = -focal_length * Y / (Z * Z)
        
        H = np.zeros((2, 9))
        H[:, :6] = H_pose
        
        # Kalman gain
        S = H @ self.P @ H.T + self.R
        K = self.P @ H.T @ np.linalg.inv(S)
        
        # Update covariance
        self.P = (np.eye(9) - K @ H) @ self.P
        
        # Update state
        delta = K @ innovation
        
        # Update pose
        delta_pose = delta[:6]
        self.T = self.T @ exp_se3(delta_pose)
        
        # Update velocity
        self.v = self.v + delta[6:9]

print("VIO-IEKF class defined")

## 6. Run the Filter

In [ ]:
# Initialize with some error
initial_error_pose = np.array([0.1, 0.1, 0.0, 0.05, 0.05, 0.0])
T0 = exp_se3(initial_error_pose)
v0 = np.array([0.8, 0.1, 0.0])
P0 = 0.1 * np.eye(9)

Q_imu = 0.01 * np.eye(9)
R_vis = 1.0 * np.eye(2)

vio = VIO_IEKF(T0, v0, P0, Q_imu, R_vis)

# Storage
T_est = np.zeros((N, 4, 4))
v_est = np.zeros((N, 3))
pos_errors = np.zeros(N)
rot_errors = np.zeros(N)

# Run filter
print("Running VIO-IEKF...")
for i in range(N):
    # Predict with IMU
    vio.predict(imu_omega[i], imu_accel[i], dt)
    
    # Update with visual measurements if available
    if i in visual_measurements:
        for landmark_idx, measurement in visual_measurements[i]:
            vio.update_visual(landmarks[landmark_idx], measurement)
    
    # Store
    T_est[i] = vio.T
    v_est[i] = vio.v
    
    # Compute errors
    pos_errors[i] = np.linalg.norm(T_true[i, :3, 3] - T_est[i, :3, 3])
    # Rotation error (simplified)
    R_error = T_true[i, :3, :3].T @ T_est[i, :3, :3]
    rot_errors[i] = np.arccos((np.trace(R_error) - 1) / 2)

print(f"Filter complete.")
print(f"Final position error: {pos_errors[-1]:.4f} m")
print(f"Final rotation error: {np.degrees(rot_errors[-1]):.2f} deg")

## 7. Visualize Results

In [ ]:
# Extract positions
pos_true = T_true[:, :3, 3]
pos_est = T_est[:, :3, 3]

# 3D trajectory plot
fig = plt.figure(figsize=(14, 5))

# 3D trajectory
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot(pos_true[:, 0], pos_true[:, 1], pos_true[:, 2], 
         'g-', linewidth=2, label='Ground Truth')
ax1.plot(pos_est[:, 0], pos_est[:, 1], pos_est[:, 2], 
         'b--', linewidth=2, label='IEKF Estimate')
ax1.scatter(landmarks[:, 0], landmarks[:, 1], landmarks[:, 2], 
           c='r', marker='x', s=100, label='Landmarks')
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_zlabel('Z (m)')
ax1.set_title('3D Trajectory')
ax1.legend()

# Position errors
ax2 = fig.add_subplot(132)
time = np.arange(N) * dt
ax2.plot(time, pos_errors, 'b-', linewidth=2)
# Mark visual measurements
for t_vis in visual_measurements.keys():
    ax2.axvline(t_vis * dt, color='r', alpha=0.2, linewidth=0.5)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Position Error (m)')
ax2.set_title('Position Error')
ax2.grid(True)

# Rotation errors
ax3 = fig.add_subplot(133)
ax3.plot(time, np.degrees(rot_errors), 'r-', linewidth=2)
for t_vis in visual_measurements.keys():
    ax3.axvline(t_vis * dt, color='r', alpha=0.2, linewidth=0.5)
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Rotation Error (deg)')
ax3.set_title('Rotation Error')
ax3.grid(True)

plt.tight_layout()
plt.show()

print(f"\nStatistics:")
print(f"Mean position error: {np.mean(pos_errors):.4f} m")
print(f"Mean rotation error: {np.degrees(np.mean(rot_errors)):.2f} deg")

## 8. Key Observations

### IEKF Performance
1. **Drift reduction**: Visual measurements correct IMU drift
2. **Consistency**: Errors remain bounded despite noisy measurements
3. **Geometric correctness**: Pose estimates stay on SE(3)

### Benefits of Invariant Formulation
- Error dynamics independent of current pose
- Better consistency than standard EKF
- Natural handling of SE(3) geometry

### Practical Considerations
- Visual measurements are sparse but highly informative
- IMU provides high-rate predictions
- Landmark association is assumed known (simplified)

## 9. Extensions and Improvements

This is a simplified VIO implementation. Real systems include:

### State Augmentation
- IMU biases (gyroscope and accelerometer)
- Camera-IMU extrinsics calibration
- Scale factor (for monocular case)

### Measurement Processing
- Feature tracking across frames
- Outlier rejection (RANSAC)
- Multiple cameras

### Advanced Techniques
- Sliding window optimization
- Loop closure detection
- Map management

### Alternative Approaches
- **Equivariant Filter**: More general than IEKF
- **MSCKF**: Multi-State Constraint Kalman Filter
- **Optimization-based**: Bundle adjustment

## 10. Comparison with Standard EKF

A standard EKF for VIO would:
- Define errors additively in R³ for position and R³ for orientation
- Risk inconsistency as errors don't respect SE(3) structure
- Require more careful tuning

The IEKF:
- Uses multiplicative errors on SE(3)
- Maintains better consistency
- More robust to initialization errors

## Exercises

1. **Add IMU biases**: Augment state with gyroscope and accelerometer biases
2. **Tune parameters**: Experiment with different noise parameters
3. **Compare trajectories**: Try different motion patterns
4. **Implement standard EKF**: Compare performance
5. **Add outliers**: Test robustness to bad visual measurements

## References

### Papers
- Barrau & Bonnabel (2017). "The Invariant Extended Kalman Filter as a Stable Observer"
- Forster et al. (2017). "On-Manifold Preintegration for Real-Time Visual-Inertial Odometry"

### Software
- [GTSAM](https://github.com/borglab/gtsam) - Factor graph optimization library
- [VINS-Mono](https://github.com/HKUST-Aerial-Robotics/VINS-Mono) - Robust VIO system
- [ORB-SLAM3](https://github.com/UZ-SLAMLab/ORB_SLAM3) - Complete SLAM system

### Documentation
- [GTSAM EKF Variants](https://borglab.github.io/gtsam/ekf-variants/)
- [AwesomeEqF Resources](https://github.com/borglab/AwesomeEqF)